# 03b SKU Classification - Standardized Input

## Purpose

This notebook is the standardized-input version of `03_sku_classification.ipynb`. It reproduces the existing SKU classification logic using the processed standardized outputs created by `01b_data_cleaning_standardized_input.ipynb`.

This is a preparation layer for future reusable pipeline refactoring. It does not replace the original workflow or change the validated project metrics.

## Input and output scope

The notebook consumes:

```text
data/processed_standardized/monthly_sku_sales.csv
data/processed_standardized/sku_master.csv
```

It writes the classification outputs only to `outputs_standardized/`. The original `data/processed/` and `outputs/` directories are not modified.

`stock_code` is the normalized SKU key. SKU profiles are grouped only by `stock_code`, and `description` is merged from the SKU master as a display field.

## Load standardized processed data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Resolve paths whether the notebook runs from the project root or notebooks/.
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_dir = project_root / "data" / "processed_standardized"
outputs_dir = project_root / "outputs_standardized"

monthly_sku_path = processed_dir / "monthly_sku_sales.csv"
sku_master_path = processed_dir / "sku_master.csv"

missing_files = [
    path for path in [monthly_sku_path, sku_master_path] if not path.exists()
]
if missing_files:
    raise FileNotFoundError(
        "Missing standardized processed input(s): "
        + ", ".join(str(path) for path in missing_files)
        + ". Run notebooks/01b_data_cleaning_standardized_input.ipynb first."
    )

monthly_sku_sales = pd.read_csv(
    monthly_sku_path,
    dtype={
        "stock_code": "string",
        "description": "string",
        "invoice_month": "string",
    },
)
sku_master = pd.read_csv(
    sku_master_path,
    dtype={"stock_code": "string", "description": "string"},
)

monthly_required = {
    "stock_code", "invoice_month", "monthly_units", "monthly_revenue",
    "order_count", "avg_unit_price",
}
master_required = {"stock_code", "description"}

missing_monthly_columns = sorted(monthly_required - set(monthly_sku_sales.columns))
missing_master_columns = sorted(master_required - set(sku_master.columns))
if missing_monthly_columns:
    raise ValueError(
        "Missing monthly SKU sales columns: " + ", ".join(missing_monthly_columns)
    )
if missing_master_columns:
    raise ValueError(
        "Missing SKU master columns: " + ", ".join(missing_master_columns)
    )
if sku_master["stock_code"].duplicated().any():
    raise ValueError("sku_master must contain one row per stock_code.")

outputs_dir.mkdir(parents=True, exist_ok=True)

print("Monthly SKU sales shape:", monthly_sku_sales.shape)
print("SKU master shape:", sku_master.shape)

## Build the SKU-level profile

The monthly table is aggregated into one profile per `stock_code`. The profile preserves notebook 03's sales volume, revenue, active-month, monthly-demand, volatility, price, and order measures. Description is then merged from `sku_master` for display.

In [ ]:
sku_profile = (
    monthly_sku_sales
    .groupby("stock_code", as_index=False)
    .agg(
        total_units=("monthly_units", "sum"),
        total_revenue=("monthly_revenue", "sum"),
        active_months=("invoice_month", "nunique"),
        avg_monthly_units=("monthly_units", "mean"),
        std_monthly_units=("monthly_units", "std"),
        avg_unit_price=("avg_unit_price", "mean"),
        total_orders=("order_count", "sum"),
    )
)

sku_profile = sku_profile.merge(
    sku_master[["stock_code", "description"]],
    on="stock_code",
    how="left",
    validate="one_to_one",
)

# Fill missing standard deviation for SKUs active in only one month.
sku_profile["std_monthly_units"] = sku_profile["std_monthly_units"].fillna(0)

# Demand volatility coefficient.
sku_profile["demand_cv"] = np.where(
    sku_profile["avg_monthly_units"] > 0,
    sku_profile["std_monthly_units"] / sku_profile["avg_monthly_units"],
    0,
)

print("SKU profile shape:", sku_profile.shape)
print("Unique stock_code values:", sku_profile["stock_code"].nunique())
sku_profile.head()

## Sales contribution thresholds

The classification uses the same percentile thresholds as notebook 03:

- Top 20% of SKUs by revenue for revenue priority.
- Top 20% of SKUs by units for high turnover.
- Bottom 30% of SKUs by units for long tail.
- 75th percentile of demand CV to separate stable and volatile high-turnover SKUs.

In [ ]:
revenue_80 = sku_profile["total_revenue"].quantile(0.80)
units_80 = sku_profile["total_units"].quantile(0.80)
units_30 = sku_profile["total_units"].quantile(0.30)
cv_75 = sku_profile["demand_cv"].quantile(0.75)

print("Revenue 80th percentile:", round(revenue_80, 2))
print("Units 80th percentile:", round(units_80, 2))
print("Units 30th percentile:", round(units_30, 2))
print("Demand CV 75th percentile:", round(cv_75, 2))

## SKU classification

Rule order and boundaries match notebook 03:

1. High-Revenue Priority
2. High-Turnover Stable
3. High-Turnover Volatile
4. Long-Tail
5. Regular

In [ ]:
def classify_sku(row):
    if row["total_revenue"] >= revenue_80:
        return "High-Revenue Priority"
    elif row["total_units"] >= units_80 and row["demand_cv"] <= cv_75:
        return "High-Turnover Stable"
    elif row["total_units"] >= units_80 and row["demand_cv"] > cv_75:
        return "High-Turnover Volatile"
    elif row["total_units"] <= units_30:
        return "Long-Tail"
    else:
        return "Regular"

sku_profile["sku_class"] = sku_profile.apply(classify_sku, axis=1)

sku_profile[
    ["stock_code", "description", "total_units", "total_revenue", "demand_cv", "sku_class"]
].head()

## Classification summary

The summary reports SKU counts, units, revenue, average demand CV, revenue share, and unit share for each class.

In [ ]:
classification_summary = (
    sku_profile
    .groupby("sku_class", as_index=False)
    .agg(
        sku_count=("stock_code", "count"),
        total_units=("total_units", "sum"),
        total_revenue=("total_revenue", "sum"),
        avg_demand_cv=("demand_cv", "mean"),
    )
    .sort_values("total_revenue", ascending=False)
)

classification_summary["revenue_share"] = (
    classification_summary["total_revenue"]
    / classification_summary["total_revenue"].sum()
)

classification_summary["unit_share"] = (
    classification_summary["total_units"]
    / classification_summary["total_units"].sum()
)

classification_summary

## Recommended inventory actions

Each class uses the same practical inventory action mapping as notebook 03.

In [ ]:
action_mapping = {
    "High-Revenue Priority": "Prioritize inventory monitoring and avoid stockouts",
    "High-Turnover Stable": "Keep stable local warehouse inventory",
    "High-Turnover Volatile": "Monitor closely and replenish in smaller batches",
    "Long-Tail": "Limit stock and avoid excessive local warehouse space",
    "Regular": "Maintain standard replenishment review",
}

sku_profile["recommended_action"] = sku_profile["sku_class"].map(action_mapping)

sku_profile[
    [
        "stock_code", "description", "total_units", "total_revenue",
        "demand_cv", "sku_class", "recommended_action",
    ]
].head(20)

## Save standardized outputs

Both outputs are written only to `outputs_standardized/`, keeping them separate from the original workflow.

In [ ]:
sku_profile_path = outputs_dir / "sku_profile_classification.csv"
classification_summary_path = outputs_dir / "sku_classification_summary.csv"

sku_profile.to_csv(sku_profile_path, index=False)
classification_summary.to_csv(classification_summary_path, index=False)

print("Saved output files:")
print("- outputs_standardized/sku_profile_classification.csv")
print("- outputs_standardized/sku_classification_summary.csv")

## Management interpretation

For the current project data, the results should remain aligned with notebook 03: 784 High-Revenue Priority SKUs contribute approximately 78.8% of revenue and 66.1% of units, while 1,172 Long-Tail SKUs contribute approximately 1.3% of revenue and 0.6% of units.

These classifications remain a preparation layer for later standardized replenishment and warehouse-allocation work.

## Final summary

The final cell lists output files and shapes, then reports SKU counts, revenue share, and unit share for every class.

In [ ]:
output_summary = {
    "outputs_standardized/sku_profile_classification.csv": sku_profile.shape,
    "outputs_standardized/sku_classification_summary.csv": classification_summary.shape,
}

print("===== 03b Standardized-Input SKU Classification Summary =====")
print("SKU profiles classified:", len(sku_profile))
print("\nOutput files and shapes:")
for output_file, shape in output_summary.items():
    print(f"- {output_file}: {shape}")

print("\nSKU class counts, revenue share, and unit share:")
for _, metrics in classification_summary.iterrows():
    sku_class = metrics["sku_class"]
    print(
        f"- {sku_class}: {int(metrics['sku_count']):,} SKUs; "
        f"revenue share {metrics['revenue_share']:.1%}; "
        f"unit share {metrics['unit_share']:.1%}"
    )